# Severity Model — Claim Amount Distribution

This notebook models the **claim severity** (payment amount) in a French MTPL portfolio.

We use a **spliced model** combining:
- A **Truncated Lognormal** for the body (below a threshold)
- A **Generalized Pareto Distribution (GPD)** for the tail (above the threshold)

The threshold is selected using the Mean Excess Plot.

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
from scipy.stats import lognorm, genpareto

df = pd.read_csv("freMTPL2sev.csv")
claim_amounts = df["ClaimAmount"]
n_payments = claim_amounts.count()

print(f"Number of payments : {n_payments}")
print(f"Min                : {claim_amounts.min():.2f}")
print(f"Q25                : {np.percentile(claim_amounts, 25):.2f}")
print(f"Q75                : {np.percentile(claim_amounts, 75):.2f}")
print(f"Q99                : {np.percentile(claim_amounts, 99):.2f}")
print(f"Q99.5              : {np.percentile(claim_amounts, 99.5):.2f}")
print(f"Q99.9              : {np.percentile(claim_amounts, 99.9):.2f}")
print(f"Max                : {claim_amounts.max():.2f}")

## 2. Threshold Selection — Mean Excess Plot

The Mean Excess Plot helps identify the threshold above which the tail follows a GPD.
A linear increase in the mean excess function indicates GPD behaviour.

In [ ]:
def mean_excess_plot(data):
    """Plot the mean excess function to help select the GPD threshold."""
    thresholds = np.percentile(data, np.arange(50, 99, 1))
    mean_excess = [data[data > t].mean() - t for t in thresholds]
    plt.plot(thresholds, mean_excess)
    plt.xlabel("Threshold")
    plt.ylabel("Mean Excess")
    plt.title("Mean Excess Plot")
    plt.show()

mean_excess_plot(claim_amounts)

## 3. GPD Shape Parameter Across Candidate Thresholds

We fit a GPD on the excess above several candidate thresholds and compare the shape parameter ξ.
A stable ξ across thresholds confirms the GPD is appropriate.

In [ ]:
candidate_thresholds = [3000, 4000, 5000, 7000]

for t in candidate_thresholds:
    excess = claim_amounts[claim_amounts > t] - t
    shape, _, _ = genpareto.fit(excess, floc=0)
    print(f"Threshold {t:>5} — GPD shape (xi) : {shape:.4f}")

## 4. Selected Threshold and Body/Tail Split

Based on the Mean Excess Plot and GPD shape stability, we select **€4,000** as the threshold.

In [ ]:
threshold = 5000

body = claim_amounts[claim_amounts <= threshold]
tail = claim_amounts[claim_amounts > threshold]
excess = tail - threshold

# Probability of a claim falling in the body
p_body = len(body) / len(claim_amounts)

print(f"Threshold          : €{threshold}")
print(f"Body size          : {len(body)} ({p_body:.4f})")
print(f"Tail size          : {len(tail)} ({1 - p_body:.4f})")
print(f"Quantile check     : €{np.quantile(claim_amounts, p_body):.2f} (should equal threshold)")

## 5. Distribution Fitting

### 5.1 Body — Truncated Lognormal

In [ ]:
shape_ln, loc_ln, scale_ln = lognorm.fit(body, floc=0)

print(f"Lognormal fit:")
print(f"  shape : {shape_ln:.4f}")
print(f"  scale : {scale_ln:.4f}")

### 5.2 Tail — Generalized Pareto Distribution

In [ ]:
shape_gpd, loc_gpd, scale_gpd = genpareto.fit(excess, floc=0)

print(f"GPD fit on excess above €{threshold}:")
print(f"  shape (xi)   : {shape_gpd:.4f}")
print(f"  scale (beta) : {scale_gpd:.4f}")

## 6. Simulation Functions

In [ ]:
def sample_lognormal_truncated(n, threshold, shape, scale):
    """Sample n values from a Lognormal distribution truncated at threshold."""
    samples = []
    while len(samples) < n:
        x = lognorm.rvs(shape, loc=0, scale=scale, size=n)
        x = x[x <= threshold]
        samples.extend(x.tolist())
    return np.array(samples[:n])


def sample_gpd(n, threshold, shape, scale):
    """Sample n values from a GPD shifted by threshold."""
    excess = genpareto.rvs(shape, loc=0, scale=scale, size=n)
    return threshold + excess


def simulate_severity(n):
    """Simulate n claim severities using the spliced Lognormal/GPD model."""
    u = np.random.uniform(size=n)

    n_body = np.sum(u < p_body)
    n_tail = n - n_body

    samples = np.zeros(n)
    samples[u < p_body] = sample_lognormal_truncated(
        n_body, threshold, shape_ln, scale_ln
    )
    samples[u >= p_body] = sample_gpd(
        n_tail, threshold, shape_gpd, scale_gpd
    )
    return samples

## 7. Model Validation

The histogram — even on a log scale — is not informative enough to validate the fit due to the large 
difference in sample sizes between data and simulation. The quantile comparison and QQ plot below 
are the primary visual validation tools.

In [ ]:
sim = simulate_severity(100000)

# Density comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normal scale
axes[0].hist(claim_amounts, bins=100, density=True, alpha=0.5, label="Data")
axes[0].hist(sim, bins=100, density=True, alpha=0.5, label="Simulation")
axes[0].set_xlim(0, 10000)
axes[0].set_title("Data vs Simulation — Density")
axes[0].legend()

# Log scale
axes[1].hist(claim_amounts, bins=100, density=True, alpha=0.5, label="Data")
axes[1].hist(sim, bins=100, density=True, alpha=0.5, label="Simulation")
axes[1].set_yscale('log')
axes[1].set_xlim(0, 10000)
axes[1].set_title("Data vs Simulation — Log Scale")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Quantile comparison
q = np.linspace(0.01, 0.99, 100)
q_data = np.quantile(claim_amounts, q)
q_sim = np.quantile(sim, q)

plt.plot(q_data, label="Data")
plt.plot(q_sim, label="Simulation")
plt.legend()
plt.title("Quantile Comparison")
plt.show()

In [ ]:
# QQ Plot
plt.scatter(q_data, q_sim, s=10)
plt.plot([0, max(q_data)], [0, max(q_data)], '--', color='red')
plt.xlabel("Data quantiles")
plt.ylabel("Simulation quantiles")
plt.title("QQ Plot — Data vs Simulation")
plt.show()

## 8. Risk Measures on Simulated Severity

In [ ]:
var_995_sim = np.quantile(sim, 0.995)
tvar_995_sim = sim[sim >= var_995_sim].mean()

var_995_data = np.quantile(claim_amounts, 0.995)
tvar_995_data = claim_amounts[claim_amounts >= var_995_sim].mean()

print("Risk Measures at 99.5%")
print(f"  VaR  99.5% — Simulation : {var_995_sim:.2f} €")
print(f"  VaR  99.5% — Data       : {var_995_data:.2f} €")
print(f"  TVaR 99.5% — Simulation : {tvar_995_sim:.2f} €")
print(f"  TVaR 99.5% — Data       : {tvar_995_data:.2f} €")

## 9. Conclusion

The spliced **Lognormal / GPD** model provides a good fit across the full severity range:
- The Lognormal captures the bulk of claims below €4,000
- The GPD captures the heavy tail above €4,000

**Known limitation** : - **Lognormal fit** : The Lognormal does not fully capture the body profile — the empirical quantile curve suggests two distinct regimes, possibly driven by structural effects in the data (fixed tariffs, coverage caps, or a mix of claim types such as material vs bodily injury). Alternative distributions such as Burr or Weibull could be investigated.

The fitted parameters (`shape_ln`, `scale_ln`, `shape_gpd`, `scale_gpd`, `p_body`, `threshold`) are carried forward into `aggregate_loss.ipynb`.